In [1]:
!pip uninstall -y peft transformers tokenizers datasets evaluate seqeval accelerate
!pip install transformers==4.46.3 peft==0.14.0 tokenizers==0.20.3 datasets==2.19.1 evaluate==0.4.2 seqeval==1.2.2 accelerate==0.34.2 -q

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import transformers
import peft

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

transformers: 4.46.3
peft: 0.14.0


In [2]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
import evaluate

print("All imports successful")

All imports successful


In [3]:
dataset = load_dataset("lhoestq/conll2003")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [4]:
print(dataset["train"][0])

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [5]:
label_list = [
    "O",
    "B-NP", "I-NP",
    "B-VP", "I-VP",
    "B-PP", "I-PP",
    "B-ADVP", "I-ADVP",
    "B-ADJP", "I-ADJP",
    "B-SBAR", "I-SBAR",
    "B-PRT", "I-PRT",
    "B-INTJ", "I-INTJ",
    "B-CONJP", "I-CONJP",
    "B-LST", "I-LST",
    "B-UCP", "I-UCP"
]

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

print("Total labels:", len(label_list))

Total labels: 23


In [6]:
example = dataset["train"][0]

tokens = example["tokens"]
labels = example["chunk_tags"]

for t, l in zip(tokens, labels):
    print(t, "→", label_list[l])

EU → B-SBAR
rejects → B-UCP
German → B-SBAR
call → I-SBAR
to → B-UCP
boycott → I-UCP
British → B-SBAR
lamb → I-SBAR
. → O


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("Tokenizer loaded")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded


In [8]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [9]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
print(tokenized_datasets["train"][0].keys())

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

dict_keys(['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'attention_mask', 'labels'])


In [10]:
print("Input IDs:", tokenized_datasets["train"][0]["input_ids"])
print("Attention Mask:", tokenized_datasets["train"][0]["attention_mask"])
print("Labels:", tokenized_datasets["train"][0]["labels"])

Input IDs: [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels: [-100, 11, 21, 11, 12, 21, 22, 11, 12, 0, -100]


In [11]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Data collator ready")

Data collator ready


In [13]:
import evaluate

metric = evaluate.load("seqeval")
print("Metric loaded")

Metric loaded


In [14]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

print("Training arguments set")

Training arguments set


In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready")

Trainer ready


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.438100,0.340564,0.832168,0.833702,0.832934,0.917895
2,0.275900,0.312315,0.856180,0.842920,0.849498,0.925439


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=500, training_loss=0.5205173263549805, metrics={'train_runtime': 47.0093, 'train_samples_per_second': 85.09, 'train_steps_per_second': 10.636, 'total_flos': 41720713401360.0, 'train_loss': 0.5205173263549805, 'epoch': 2.0})

In [18]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.312315434217453, 'eval_precision': 0.8561797752808988, 'eval_recall': 0.8429203539823009, 'eval_f1': 0.8494983277591973, 'eval_accuracy': 0.9254385964912281, 'eval_runtime': 1.1731, 'eval_samples_per_second': 426.21, 'eval_steps_per_second': 53.702, 'epoch': 2.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [21]:
import torch
import numpy as np

sentence = "John works at Google in California"
tokens = sentence.split()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs).logits

predictions = torch.argmax(outputs, dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0][1:len(tokens)+1]]

for token, label in zip(tokens, predicted_labels):
    print(token, "->", label)

John -> B-SBAR
works -> B-UCP
at -> B-PRT
Google -> B-SBAR
in -> B-PRT
California -> B-SBAR


In [22]:
num_train_epochs=3
train_dataset=tokenized_datasets["train"].select(range(5000))

In [23]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.220400,0.284781,0.884789,0.866519,0.875559,0.937368
2,0.157600,0.255565,0.884372,0.874263,0.879288,0.940877
3,0.133500,0.248938,0.892245,0.882375,0.887282,0.946316


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=1875, training_loss=0.1708291697184245, metrics={'train_runtime': 122.5695, 'train_samples_per_second': 122.38, 'train_steps_per_second': 15.297, 'total_flos': 153404136661440.0, 'train_loss': 0.1708291697184245, 'epoch': 3.0})

In [24]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.24893759191036224, 'eval_precision': 0.8922445935868755, 'eval_recall': 0.8823746312684366, 'eval_f1': 0.8872821653689285, 'eval_accuracy': 0.9463157894736842, 'eval_runtime': 1.8532, 'eval_samples_per_second': 269.805, 'eval_steps_per_second': 33.995, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [25]:
import torch
import numpy as np

sentence = "John works at Google in California"
tokens = sentence.split()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs).logits

predictions = torch.argmax(outputs, dim=2)
predicted_labels = [label_list[p.item()] for p in predictions[0][1:len(tokens)+1]]

for token, label in zip(tokens, predicted_labels):
    print(token, "->", label)

John -> B-SBAR
works -> B-UCP
at -> B-PRT
Google -> B-SBAR
in -> B-PRT
California -> B-SBAR


POS tagging assigns grammatical labels to individual words such as noun, verb, adjective, and pronoun.

Chunking groups words into meaningful phrases such as noun phrases, verb phrases, and prepositional phrases.

POS tagging is easier because it works at the word level, while chunking is more complex because it captures phrase-level structure.

 **Conclusion**
 In this assignment, I fine-tuned a DistilBERT model for chunking using the CoNLL-2003 dataset. I performed tokenization, label alignment, model setup, training, evaluation, and inference.

A key challenge in this task was aligning labels correctly with tokenized subwords and handling special tokens using -100. This task helped me understand how transformer models can be used for token classification tasks such as POS tagging and chunking.